In [17]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
load_dotenv()

True

In [18]:
eval_message = ["""I asked someone to answer a question based on one or more documents.
Your task is to review their response and assess whether or not each sentence
in that response is supported by text in the documents. And if so, which
sentences in the documents provide that support. You will also tell me which
of the documents contain useful information for answering the question, and
which of the documents the answer was sourced from.
Here are the documents, each of which is split into sentences. Alongside each
sentence is associated key, such as ’a0.’ or ’b0.’ that you can use to refer
to it:
‘‘‘
{documents}
‘‘‘
The question was:
‘‘‘
{question}
‘‘‘
Here is their response:
‘‘‘
{answer}
‘‘‘
You must respond with a JSON object matching this schema:
‘‘‘
{{
"relevance_explanation": string,
"all_relevant_sentence_keys": [string],
"overall_supported_explanation": string,
"overall_supported": boolean,
"sentence_support_information": [
{{
"response_sentence_key": string,
"explanation": string,
    "supporting_sentence_keys": [string],
"fully_supported": boolean
}}
],
"all_utilized_sentence_keys": [string]
}}
‘‘‘
The relevance_explanation field is a string explaining which documents
contain useful information for answering the question. Provide a step-by-step
breakdown of information provided in the documents and how it is useful for
answering the question.
The all_relevant_sentence_keys field is a list of all document sentences keys
(e.g. ’a0’) that are revant to the question. Include every sentence that is
useful and relevant to the question, even if it was not used in the response,
or if only parts of the sentence are useful. Ignore the provided response when
making this judgement and base your judgement solely on the provided documents
and question. Omit sentences that, if removed from the document, would not
impact someone’s ability to answer the question.
The overall_supported_explanation field is a string explaining why the response
*as a whole* is or is not supported by the documents. In this field, provide a
step-by-step breakdown of the claims made in the response and the support (or
lack thereof) for those claims in the documents. Begin by assessing each claim
separately, one by one; don’t make any remarks about the response as a whole
until you have assessed all the claims in isolation.
The overall_supported field is a boolean indicating whether the response as a
whole is supported by the documents. This value should reflect the conclusion
you drew at the end of your step-by-step breakdown in overall_supported_explanation.
In the sentence_support_information field, provide information about the support
*for each sentence* in the response.
The sentence_support_information field is a list of objects, one for each sentence
in the response. Each object MUST have the following fields:
- response_sentence_key: a string identifying the sentence in the response.
This key is the same as the one used in the response above.
- explanation: a string explaining why the sentence is or is not supported by the
documents.
- supporting_sentence_keys: keys (e.g. ’a0’) of sentences from the documents that
support the response sentence. If the sentence is not supported, this list MUST
be empty. If the sentence is supported, this list MUST contain one or more keys.
In special cases where the sentence is supported, but not by any specific sentence,
you can use the string "supported_without_sentence" to indicate that the sentence
is generally supported by the documents. Consider cases where the sentence is
expressing inability to answer the question due to lack of relevant information in
the provided contex as "supported_without_sentence". In cases where the sentence
is making a general statement (e.g. outlining the steps to produce an answer, or
summarizing previously stated sentences, or a transition sentence), use the
string "general". In cases where the sentence is correctly stating a well-known fact,
like a mathematical formula, use the string "well_known_fact". In cases where the
sentence is performing numerical reasoning (e.g. addition, multiplication), use
the string "numerical_reasoning".
- fully_supported: a boolean indicating whether the sentence is fully supported by
the documents.
- This value should reflect the conclusion you drew at the end of your step-by-step
breakdown in explanation.
- If supporting_sentence_keys is an empty list, then fully_supported must be false.
- Otherwise, use fully_supported to clarify whether everything in the response
sentence is fully supported by the document text indicated in supporting_sentence_keys
(fully_supported = true), or whether the sentence is only partially or incompletely
supported by that document text (fully_supported = false).
The all_utilized_sentence_keys field is a list of all sentences keys (e.g. ’a0’) that
were used to construct the answer. Include every sentence that either directly supported
the answer, or was implicitly used to construct the answer, even if it was not used
in its entirety. Omit sentences that were not used, and could have been removed from
the documents without affecting the answer.
You must respond with a valid JSON string. Use escapes for quotes, e.g. ‘\\"‘, and
newlines, e.g. ‘\\n‘. Do not write anything before or after the JSON string. Do not
wrap the JSON string in backticks like ‘‘‘ or ‘‘‘json.
As a reminder: your task is to review the response and assess which documents contain
useful information pertaining to the question, and how each sentence in the response
is supported by the text in the documents.
"""]

In [19]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json
import re
from json_repair import repair_json

In [20]:
def simple_rag(query: str, eval_message: list, expert_domain: str, retriever, gen_model="llama-3.1-8b-instant", evaluate_model="llama-3.3-70b-versatile",
              model_type="groq"):
    messages = [
    ("system", """You are a {expert_domain} expert. You are given a question and a list of documents and need to
    answer the question. Answer the question only based on these documents. These
    documents can help you answer the question: {context}. If you are not sure about the
    answer, you can say 'I don't know' or 'I don't know the answer to that question.'"""),
    ("human", "{question}"),
    ]

    # db_path = db_path
    # vector_db = Chroma(persist_directory=db_path, embedding_function=HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5"))


    # retriever = vector_db.as_retriever(search_type='similarity', search_kwargs={"k":5})
    relevant_docs = retriever.invoke(query)
    context = "\n".join([d.page_content for d in relevant_docs])
    # print(relevant_docs)
    sent_list = list()
    for i, d in enumerate(relevant_docs):
        index_char = chr(97+i)
        for j, s in enumerate(d.page_content.split(".")):
            sent_list.append([index_char+str(j), s])
    

    q_prompt = ChatPromptTemplate(messages=messages)
    q_model = ChatGroq(model=gen_model, temperature=0)
    q_chain = q_prompt | q_model | StrOutputParser()

    response = q_chain.invoke({"question": query, "context": context, "expert_domain": expert_domain})

    eval_prompt = ChatPromptTemplate(messages=eval_message)
    if model_type == "ollama":
        eval_model = ChatOllama(model=evaluate_model, temperature=0)
    elif model_type == "groq":
        eval_model = ChatGroq(model=evaluate_model, temperature=0)
    eval_chain = eval_prompt | eval_model | StrOutputParser()

    eval_response = eval_chain.invoke({"documents":sent_list, "question":query, "answer":response})
    # print(response)
    # print(eval_response)

    return (response, eval_response, sent_list)

    
    

    

    
        
    
    

In [21]:
def get_metrics(e, sl, r):
    relevance = 0.0
    utilization = 0.0
    adherence = 0.0
    completeness = 0.0
    # print(e)
    match = re.search(r'\{.*\}', e, re.DOTALL)
    if match:
        e_raw = match.group(0)
    else:
        e_raw = e
    try:
        e = json.loads(repair_json(e_raw))
    except Exception as err:
        print(f"Critical Parsing Error: {err}")

    total_sentences_len = len(sl)
    len_all_relevant_sentence_keys = len(e["all_relevant_sentence_keys"])
    len_all_utilized_sentence_keys = len(e["all_utilized_sentence_keys"])
    relevant_set = set(e["all_relevant_sentence_keys"])
    utilized_set = set(e["all_utilized_sentence_keys"])
    intersect_keys = relevant_set.intersection(utilized_set)
    len_intersect_keys = len(intersect_keys)
    sentence_support_information = [s.get("fully_supported") for s in e["sentence_support_information"]]
    
    relevance = len_all_relevant_sentence_keys/total_sentences_len if total_sentences_len > 0 else 0.0
    utilization = len_all_utilized_sentence_keys/total_sentences_len if total_sentences_len > 0 else 0.0
    completeness = len_intersect_keys/len_all_relevant_sentence_keys if len_all_relevant_sentence_keys > 0 else 0.0
    adherence = int(all(sentence_support_information))

    return relevance, utilization, completeness, adherence
    
    # print(relevance, utilization, completeness, sentence_support_information, adherence)

In [55]:
def get_output(query: str, eval_message: list, retriever, expert_domain: str = "finance", gen_model="llama-3.1-8b-instant", evaluate_model="llama-3.3-70b-versatile", model_type="ollama"):
    output = simple_rag(query=query, eval_message=eval_message, expert_domain=expert_domain, retriever=retriever, gen_model=gen_model, evaluate_model=evaluate_model, model_type=model_type)
    r, e, sl = output
    relevance, utilization, completeness, adherence = get_metrics(e, sl, r)
    # print(f"""Question: {query}\nAnswer: {r}\n\n{'='*50}\n\nScores: \nAdherence: {adherence}\nRelevance: {relevance:.2f}\nUtilization: {utilization:.2f}\nCompleteness: {completeness:.2f}
    # \n\n{'='*50}\n\noutput_evalution: \n{e}""")
    return (relevance, utilization, completeness, adherence)

    
    

In [88]:
# adherence_score = [int(value) for value in ragbench_finqa["adherence_score"]]
# relevance_score = [value for value in ragbench_finqa["relevance_score"]]
# utilization_score = [value for value in ragbench_finqa["utilization_score"]]
# completeness_score = [value for value in ragbench_finqa["completeness_score"]]
# questions= [value for value in ragbench_finqa["question"]]

In [ ]:
# adherence_score_mean = np.mean(adherence_score)
# relevance_score_mean = np.mean(relevance_score)
# utilization_score_mean = np.mean(utilization_score)
# completeness_score_mean = np.mean(completeness_score)

# print(f"adherence_score_mean: {adherence_score_mean:.2f}")
# print(f"relevance_score_mean: {relevance_score_mean:.2f}")
# print(f"utilization_score_mean: {utilization_score_mean:.2f}")
# print(f"completeness_score_mean: {completeness_score_mean:.2f}")

In [23]:


def calculate_metrics(query_list: list, eval_message: list, retriever, expert_domain: str = "finance", gen_model="llama-3.1-8b-instant", evaluate_model="llama-3.3-70b-versatile", 
                     model_type="ollama"):
    adherence_score = list()
    relevance_score = list()
    utilization_score = list()
    completeness_score = list()
    num_queries = len(query_list)
    print(f"Total no. of queries: {num_queries}")

    for i, q in enumerate(query_list, start=1):
        r, u, c, a = get_output(query=q, eval_message=eval_message, expert_domain=expert_domain, retriever=retriever, evaluate_model=evaluate_model, model_type=model_type)
        relevance_score.append(r)
        utilization_score.append(u)
        completeness_score.append(c)
        adherence_score.append(a)
        print(f"{i}/{num_queries} completed")

    rag_adherence_score_mean = np.mean(adherence_score)
    rag_relevance_score_mean = np.mean(relevance_score)
    rag_utilization_score_mean = np.mean(utilization_score)
    rag_completeness_score_mean = np.mean(completeness_score)
    print(f"rag_adherence_score_mean: {rag_adherence_score_mean:.2f}")
    print(f"rag_relevance_score_mean: {rag_relevance_score_mean:.2f}")
    print(f"rag_utilization_score_mean: {rag_utilization_score_mean:.2f}")
    print(f"rag_completeness_score_mean: {rag_completeness_score_mean:.2f}")
        

    

In [24]:
db_path = 'database/finance'
vector_db = Chroma(persist_directory=db_path, embedding_function=HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5"))
retriever = vector_db.as_retriever(search_type='similarity', search_kwargs={"k":5})
    


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [53]:
query = "what is the maximum depreciation rate that can be used for furniture fixtures and equipment?"
for d in ragbench_finqa:
    if d["question"] == query:
        print(f"q: {d["question"] }")
        print(f"r: {d["response"]}")
        print(f"scores: {
        d["adherence_score"],
        d["relevance_score"],
        d["utilization_score"],
        d["completeness_score"]
        
        }")

    # d["question"] for d in ragbench_finqa]

q: what is the maximum depreciation rate that can be used for furniture fixtures and equipment?
r: The maximum depreciation rate that can be used for furniture fixtures and equipment is 10 years.
scores: (True, 0.058823529411764705, 0.058823529411764705, 1.0)
q: what is the maximum depreciation rate that can be used for furniture fixtures and equipment?
r: According to the context provided, the company provides for depreciation and amortization on a straight-line basis over the following estimated useful lives:

- Land improvements: 20 years
- Buildings: 39 - 40 years
- Furniture, fixtures and equipment: 3 - 10 years

The maximum depreciation rate that can be used for furniture, fixtures, and equipment is 1/3 or 33.33% per year, as the estimated useful life for these assets is 3 to 10 years.
scores: (True, 0.058823529411764705, 0.11764705882352941, 1.0)


In [54]:


# simple_rag(query=query, eval_message=eval_message, expert_domain="finance")

get_output(query=query, eval_message=eval_message, retriever=retriever, evaluate_model="meta-llama/llama-4-scout-17b-16e-instruct", model_type="groq")
#"llama-3.3-70b-versatile"
#(0.11627906976744186, 0.046511627906976744, 0.4, 0)
#"qwen2.5:7b"
# (0.06976744186046512, 0.046511627906976744, 0.3333333333333333, 1)



Question: what is the maximum depreciation rate that can be used for furniture fixtures and equipment?
Answer: According to the provided documents, the estimated useful lives for furniture, fixtures, and equipment are 3-10 years. 

To determine the maximum depreciation rate, we need to use the minimum useful life, which is 3 years. 

The maximum depreciation rate would be 100% / 3 years = 33.33% per year.


Scores: 
Adherence: 0
Relevance: 0.05
Utilization: 0.05
Completeness: 1.00
    


output_evalution: 
```
{
  "relevance_explanation": "The documents contain information about the depreciation policies and estimated useful lives of various assets, including furniture, fixtures, and equipment. Documents 'a' and 'e' provide relevant information about the estimated useful lives of furniture, fixtures, and equipment, which is crucial for determining the maximum depreciation rate.",
  "all_relevant_sentence_keys": ["a2", "e3"],
  "overall_supported_explanation": "The response claims that 

(0.05, 0.05, 1.0, 0)

In [25]:
import random
def get_scores(queries: list, eval_message, retriever,no_of_samples=1, evaluate_model="qwen2.5:7b", model_type="ollama"):
    qsubset = random.sample(queries, no_of_samples)
    print(qsubset)
    adherence_score = list()
    relevance_score = list()
    utilization_score = list()
    completeness_score = list()
    for i, q in enumerate(ragbench_finqa["question"]):
        if q in qsubset and ragbench_finqa[i]['generation_model_name']=="gpt-3.5-turbo-0125":
            # print(q)
            # print(ragbench_finqa[i]['generation_model_name'])
            adherence_score.append(int(ragbench_finqa[i]["adherence_score"]))
            relevance_score.append(ragbench_finqa[i]["relevance_score"])
            utilization_score.append(ragbench_finqa[i]["utilization_score"])
            completeness_score.append(ragbench_finqa[i]["completeness_score"])
    adherence_score_mean = np.mean(adherence_score)
    relevance_score_mean = np.mean(relevance_score)
    utilization_score_mean = np.mean(utilization_score)
    completeness_score_mean = np.mean(completeness_score)
    print(f"adherence_score_mean: {adherence_score_mean:.2f}")
    print(f"relevance_score_mean: {relevance_score_mean:.2f}")
    print(f"utilization_score_mean: {utilization_score_mean:.2f}")
    print(f"completeness_score_mean: {completeness_score_mean:.2f}")
    print("="*100)

    calculate_metrics(query_list=qsubset, eval_message=eval_message, retriever=retriever, evaluate_model=evaluate_model, model_type=model_type)
    


In [26]:
from datasets import load_dataset
import numpy as np


In [27]:
ragbench_finqa = load_dataset("rungalileo/ragbench", "finqa", split="test")

In [40]:
# for d in ragbench_finqa:
questions = [d["question"] for d in ragbench_finqa]
questions = list(set(questions))

In [41]:
len(questions)

1138

In [42]:
questions[:3]

['what is the percent change of the amount of collateral held for indemnified securities between 2006 and 2007?',
 'based on the table , what would be the annual percent return for the companies investments?',
 'what would the cash expense for product warranties be in 2007 if the amounts increased the same percentage as in 2006 ( in millions ) ?']

In [57]:
evaluate_model = "llama-3.3-70b-versatile"
model_type = "groq"


get_scores(queries=questions, eval_message=eval_message, retriever=retriever,
           no_of_samples=10, evaluate_model=evaluate_model,
          model_type=model_type)


#adherence_score_mean: 0.80
# relevance_score_mean: 0.09
# utilization_score_mean: 0.06
# completeness_score_mean: 0.82


# rag_adherence_score_mean: 0.70
# rag_relevance_score_mean: 0.20
# rag_utilization_score_mean: 0.06
# rag_completeness_score_mean: 0.47

['what is the percentage change in the total fair value of non-vested shares from 2009 to 2010?', 'in 2010 what was the percent of the contractual obligations by year long-term debt obligations to the total', 'what was the percent of the firm 2019s total pledged assets in 2010 that was loans', 'what is the after-tax share-based compensation cost in 2010?', 'what was the 2006 tax expense?', 'what was the change in million of the unrecognized tax benefits between 2017 and 2018?', 'what is the average payment volume per transaction for visa inc?', "what is the range , in thousands , for united states' revenue from 2010-2012?", 'what was the percentage change in net derivative liabilities under bilateral agreements between 2011 and 2012?', 'how many shares were issued during the period of 2016 to 2018 , in millions?']
adherence_score_mean: 1.00
relevance_score_mean: 0.05
utilization_score_mean: 0.05
completeness_score_mean: 1.00
Total no. of queries: 10
1/10 completed
2/10 completed
3/10 c